### 提取商品信息

In [1]:
import pandas as pd

product_data_path = r"Amazon-Reviews-2023\raw_meta_fashion\meta_Amazon_Fashion.parquet"

# 如果出现 engine 错误，可以指定 engine='pyarrow' 或 'fastparquet'
df_product = pd.read_parquet(product_data_path, engine="pyarrow")

print("形状:", df_product.shape)
print("列名:", df_product.columns.tolist())
# print(df_product.head(5))
df_product.head(5)

形状: (826108, 14)
列名: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together']


,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,AMAZON FASHION,YUEDGE 5 Pairs Men's Moisture Control Cushione...,4.6,16,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],GiveGift,[],{'Package Dimensions': '10.31 x 8.5 x 1.73 inc...,B08BHN9PK5,NaN
1,AMAZON FASHION,DouBCQ Women's Palazzo Lounge Wide Leg Casual ...,4.1,7,"[Drawstring closure, Machine Wash]",[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],DouBCQ,[],{'Package Dimensions': '15 x 10.2 x 0.4 inches...,B08R39MRDW,NaN
2,AMAZON FASHION,Pastel by Vivienne Honey Vanilla Girls' Trapez...,4.3,11,"[Zipper closure, Hand Wash Only]",[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Pastel by Vivienne,[],{'Package Dimensions': '8.98 x 7.95 x 0.98 inc...,B077KJHCJ4,NaN
3,AMAZON FASHION,Mento Streamtail,2.0,1,"[Thermoplastic Rubber sole, High Density Premi...",[Slip on the Women's Mento and you're ready to...,29.81,[{'thumb': 'https://m.media-amazon.com/images/...,[],Guy Harvey,[],{'Package Dimensions': '11.22 x 4.72 x 4.33 in...,B0811M2JG9,NaN
4,AMAZON FASHION,RONNOX Women's 3-Pairs Bright Colored Calf Com...,4.3,3032,"[Pull On closure, Size Guide: ""S"" fits calf 10...",[Ronnox Calf Sleeves - Allowing Your Body to P...,17.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'HONEST Review: RONNOX Women's 3-Pa...,RONNOX,[],{'Package Dimensions': '7.7 x 4.3 x 1.8 inches...,B07SB2892S,NaN


In [ ]:
# 1) 查看列名，确认标题列
print(df_product.columns.tolist())

# 2) 自动选择可能的标题列（若未找到请替换成真实列名）
title_col = next((c for c in df_product.columns if 'title' in c.lower()), None)
if title_col is None:
    raise ValueError("未找到标题列，请手动指定 title_col")

# 3) 英文关键词正则（非捕获组）
title = df_product[title_col].fillna('').astype(str)

# 各类别正则（按优先级：AI -> Sunglasses -> Lens -> Eyeglasses）
pattern_ai = r'\b(?:ai\s*glass(?:es)?|smart\s*glass(?:es)?|smartglasses|ar\s*glass(?:es)?|augmented\s*reality|mixed\s*reality|mr\s*glass(?:es)?)\b'
pattern_sun = r'\b(?:sunglass(?:es)?|polariz(?:ed|ing)|sunnies)\b'
pattern_lens = r'\b(?:lens(?:es)?|contact\s*lens|lenses)\b'
pattern_eye = r'\b(?:eyeglass(?:es)?|glasses|spectacle(?:s)?|prescription|reading|bifocal|progressive|rimless)\b'
# accessory（配件）正则：清洁布、眼镜盒、及其它工具/配件
pattern_accessory = r'\b(?:cleaning\s*(?:cloth|cloths)|microfiber\s*(?:cloth|cloths)|glasses?\s*cloths?|lens\s*cleaning\s*cloth|eyeglass\s*case|glasses?\s*case(?:s)?|sunglasses?\s*case|hard\s*case|soft\s*case|protective\s*case|case(?:s)?|repair\s*kit|repair\s*tools|screwdriver(?:s)?|mini\s*screwdriver|nose\s*pads|nosepad|nose-pad|temple\s*tips|cleaning\s*kit|lens\s*cleaner|anti-?fog(?:ging)?|adjustment\s*tool|eyeglass(?:s)?\s*tool(?:s)?)\b'


mask_ai = title.str.contains(pattern_ai, case=False, na=False, regex=True)
mask_sun = title.str.contains(pattern_sun, case=False, na=False, regex=True) & ~mask_ai
mask_lens = title.str.contains(pattern_lens, case=False, na=False, regex=True) & ~mask_ai & ~mask_sun
mask_eye = title.str.contains(pattern_eye, case=False, na=False, regex=True) & ~mask_ai & ~mask_sun & ~mask_lens
# accessory mask：排除已被其他类别捕获的项（保持优先级不变）
mask_accessory = title.str.contains(pattern_accessory, case=False, na=False, regex=True) & ~mask_ai & ~mask_sun & ~mask_lens & ~mask_eye

df_ai = df_product[mask_ai].copy()
df_sunglasses = df_product[mask_sun].copy()
df_lens = df_product[mask_lens].copy()
df_eyeglasses = df_product[mask_eye].copy()
df_accessory = df_product[mask_accessory].copy()


print(f"AI Glasses: {df_ai.shape[0]}, Sunglasses: {df_sunglasses.shape[0]}, Lenses: {df_lens.shape[0]}, Eyeglasses: {df_eyeglasses.shape[0]},")

['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together']
AI Glasses: 6, Sunglasses: 20697, Lenses: 2471, Eyeglasses: 7035,


In [3]:
df_accessory

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
230,AMAZON FASHION,"Moto E (1st Gen) Case, Capsule-Case Slim Fit S...",5.0,1,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Capsule Case,[],"{'Package Dimensions': None, 'Item model numbe...",B01N2HTZYP,NaN
314,AMAZON FASHION,Canvaslove Blue Leopard Patten Canvas Laptop S...,3.3,3,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Canvaslove,[],{'Package Dimensions': '17.6 x 13.4 x 1.6 inch...,B01A5OH3CS,NaN
415,AMAZON FASHION,OutTop Holloween Pattern Pillowcase Cotton Lin...,4.4,2,"[Zipper closure, Machine Wash]",[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],OutTop,[],"{'Package Dimensions': None, 'Item model numbe...",B01JUJO6EM,NaN
579,AMAZON FASHION,"Iphone 6/6S Wallet Phone Case, Phone Card Case...",2.9,2,"[Compatible with iphone 6/6S, Color:Brown, Imp...",[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],FLY HAWK,[],{'Package Dimensions': '5.12 x 2.28 x 0.59 inc...,B06XWWJ7PK,NaN
608,AMAZON FASHION,Coach Mens Embossed Money Clip Card Case 74418...,4.0,1,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],COACH,[],{'Package Dimensions': '10.1 x 5.5 x 0.3 inche...,B00BOVBR92,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
825450,AMAZON FASHION,Diamond Rock Crystal Rhinestone Bling Hybrid B...,3.9,5,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],ATOM,[],{'Package Dimensions': '6.69 x 3.7 x 0.91 inch...,B01NA02PQE,NaN
825457,AMAZON FASHION,"Dog iColor 16.5"" 17"" 17.3"" 18"" Laptop Case Pro...",5.0,2,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],None,[],"{'Package Dimensions': None, 'Item model numbe...",B0199GDFSO,NaN
825710,AMAZON FASHION,"SERIOU Swim Goggles, Swimming Goggles No Leaki...",4.4,6,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Detaor,[],{'Package Dimensions': '7.64 x 2.36 x 1.4 inch...,B07H7DSRCZ,NaN
825720,AMAZON FASHION,"ZTE Grand X3 Case, DuroCase Transforma Kicksta...",1.0,1,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],DuroCase,[],"{'Package Dimensions': None, 'Item model numbe...",B01DN8L2RK,NaN


### 根据商品信息，找到对应的评论和投诉

In [ ]:
import pandas as pd

review_data_path = r"Amazon-Reviews-2023\raw_meta_fashion\Amazon_Fashion_review.parquet"

# 如果出现 engine 错误，可以指定 engine='pyarrow' 或 'fastparquet'
df_reviews = pd.read_parquet(review_data_path, engine="pyarrow")

print("形状:", df_reviews.shape)
print("列名:", df_reviews.columns.tolist())
df_reviews

形状: (2500939, 10)
列名: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5,Pretty locket,I think this locket is really pretty. The insi...,[],B00LOPVX74,B00LOPVX74,AGBFYI2DDIKXC5Y4FARTYDTQBMFQ,2020-01-09 00:06:34.489,3,True
1,5,A,Great,[],B07B4JXK8D,B07B4JXK8D,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,2020-12-20 01:04:06.701,0,True
2,2,Two Stars,One of the stones fell out within the first 2 ...,[],B007ZSEQ4Q,B007ZSEQ4Q,AHITBJSS7KYUBVZPX7M2WJCOIVKQ,2015-05-23 01:33:48.000,3,True
3,1,Won’t buy again,Crappy socks. Money wasted. Bought to wear wit...,[],B07F2BTFS9,B07F2BTFS9,AFVNEEPDEIH5SPUN5BWC6NKL3WNQ,2018-12-31 20:57:27.095,2,True
4,5,I LOVE these glasses,I LOVE these glasses! They fit perfectly over...,[],B00PKRFU4O,B00XESJTDE,AHSPLDNW5OOUK2PLH7GXLACFBZNQ,2015-08-13 14:29:26.000,0,True
...,...,...,...,...,...,...,...,...,...,...
2500934,5,... allowed them to be used to add military ri...,The tie tacks were the size that allowed them ...,[],B00YGFMQC0,B00YGFMQC0,AFXSFD3FTZ2CLN3TYV4B63CQM5BQ,2016-06-24 20:12:38.000,0,True
2500935,1,Didn’t come with all ten,Says ten tie clips but o only received 7.,[],B00YGFMQC0,B00YGFMQC0,AEH7WP5HGM6FGLSSC6GSTYUXBHGQ,2018-05-08 17:05:05.585,0,True
2500936,3,Not checked for quality,When I received them 2-3 of them did not open ...,[],B00YGFMQC0,B00YGFMQC0,AEL2TSSBVLIPWQ7YVMK364DUYURQ,2016-12-17 22:28:31.000,0,True
2500937,5,Awesome,Great product.,[],B00YGFMQC0,B00YGFMQC0,AGZ6IIYSPCW4YXWH6VFEOI7MTBZA,2017-04-15 17:34:26.000,1,True


In [5]:
# 1) 如果 df_reviews 没有 parent_asin,则抛出异常
if 'parent_asin' not in df_product.columns or 'parent_asin' not in df_reviews.columns:
    raise ValueError("需要 df_product 和 df_reviews 都包含 parent_asin 列（不做填充）")

# 2) 为每个类别准备 parent_asin 的集合
parents_ai    = set(df_ai['parent_asin'].dropna().unique())
parents_sun   = set(df_sunglasses['parent_asin'].dropna().unique())
parents_lens  = set(df_lens['parent_asin'].dropna().unique())
parents_eye   = set(df_eyeglasses['parent_asin'].dropna().unique())

# 3) 按 parent_asin 筛出对应 reviews
reviews_ai = df_reviews[df_reviews['parent_asin'].isin(parents_ai)].copy()
reviews_sunglasses = df_reviews[df_reviews['parent_asin'].isin(parents_sun)].copy()
reviews_lens = df_reviews[df_reviews['parent_asin'].isin(parents_lens)].copy()
reviews_eyeglasses = df_reviews[df_reviews['parent_asin'].isin(parents_eye)].copy()

# 4) 简要检查
print("reviews_ai:", len(reviews_ai))
print("reviews_sunglasses:", len(reviews_sunglasses))
print("reviews_lens:", len(reviews_lens))
print("reviews_eyeglasses:", len(reviews_eyeglasses))

reviews_ai: 15
reviews_sunglasses: 94755
reviews_lens: 10226
reviews_eyeglasses: 25205


### 合并 df_product 和 df_review 

In [ ]:
# 简洁合并函数：reviews <- products（按 parent_asin，多对一）
def merge_reviews_with_products(reviews, products, key='parent_asin', prod_prefix='prod_'):
    if key not in reviews.columns or key not in products.columns:
        raise ValueError(f"需要两表均包含列 '{key}'")
    prod_cols = [c for c in products.columns if c != key]
    prod_one = products.drop_duplicates(subset=key).set_index(key)[prod_cols]
    prod_one = prod_one.add_prefix(prod_prefix)
    merged = reviews.join(prod_one, on=key, how='left')
    return merged

# 对四类分别合并
merged_ai = merge_reviews_with_products(reviews_ai, df_ai)
merged_sunglasses = merge_reviews_with_products(reviews_sunglasses, df_sunglasses)
merged_lens = merge_reviews_with_products(reviews_lens, df_lens)
merged_eyeglasses = merge_reviews_with_products(reviews_eyeglasses, df_eyeglasses)

# 检查
print("merged_ai:", merged_ai.shape)
print("merged_sunglasses:", merged_sunglasses.shape)
print("merged_lens:", merged_lens.shape)
print("merged_eyeglasses:", merged_eyeglasses.shape)

# 查看样例
merged_ai

merged_ai: (15, 23)
merged_sunglasses: (94755, 23)
merged_lens: (10226, 23)
merged_eyeglasses: (25205, 23)


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,...,prod_rating_number,prod_features,prod_description,prod_price,prod_images,prod_videos,prod_store,prod_categories,prod_details,prod_bought_together
262501,3,Works,It does work and I do recommend it.,[],B08GQNBN9V,B08GQNBN9V,AFAOAXQ5CC46LHQW5BQZ2O4NPVIQ,2021-03-20 12:06:07.162,0,True,...,3,[non polarized],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],LINPING,[],{'Package Dimensions': '7 x 4 x 2.5 inches; 5....,NaN
431466,5,They work wonderfully,Great glasses work brilliantly and for a reaso...,[],B08GS3D8Y9,B08GS3D8Y9,AH2HQBJ4QCZQ5FA7F5Q3H3IHASBA,2020-12-24 18:25:46.654,0,True,...,17,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],LINPING,[],"{'Package Dimensions': None, 'Item model numbe...",NaN
450770,1,Didn’t hold up,Ordered in September and initially loved them....,[],B09Z2V7QV3,B09Z2V7QV3,AFWBQXUKVTEAYACHQHJX6FA7GKHQ,2022-12-16 00:48:36.147,0,False,...,2,"[Anti-Fog Coating coating, Bridge: 18 millimet...",[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],TUGAU,[],{'Package Dimensions': '7.8 x 3.23 x 2.68 inch...,NaN
820344,1,must be worn in place,not working well,[],B08GS3D8Y9,B08GS3D8Y9,AEZJEWS7CQ56S7IDHKFFH73XGSHA,2020-11-14 03:05:22.226,0,True,...,17,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],LINPING,[],"{'Package Dimensions': None, 'Item model numbe...",NaN
1021182,5,Looks good,The glasses look good and block sunlight well,[],B09GVF9BMS,B09GVF9BMS,AHGQ7TYVPSM7W4SQPAZJI4YEEYDQ,2022-07-29 20:57:20.832,0,True,...,32,"[Imported, Plastic frame, Nylon lens, Polarize...",[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Unboxing and Review of Soundcore F...,Soundcore,[],"{'Package Dimensions': None, 'Item model numbe...",NaN
1200229,2,"Don’t order a combo pack with extra lenses, th...",Sound quality is mediocre at best. Don’t order...,[],B09GVF9BMS,B09GVF9BMS,AE2EW47S5SXBC2FA42NVULKERSPQ,2022-02-16 16:54:13.618,2,True,...,32,"[Imported, Plastic frame, Nylon lens, Polarize...",[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Unboxing and Review of Soundcore F...,Soundcore,[],"{'Package Dimensions': None, 'Item model numbe...",NaN
1224551,4,Quize probar por ser algo nuevo.,Precio.Me queda bien.,[],B08GS3D8Y9,B08GS3D8Y9,AGQDPIZ2RQ6FSD6FV7MNUJPIY2KQ,2020-11-28 10:28:15.695,0,True,...,17,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],LINPING,[],"{'Package Dimensions': None, 'Item model numbe...",NaN
1232722,1,Professional product,I am very happy with the product,[],B09GVF9BMS,B09GVF9BMS,AEJUZG7U7AWOYUX3X62MUBXG4EZQ,2022-11-01 03:13:02.864,0,True,...,32,"[Imported, Plastic frame, Nylon lens, Polarize...",[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Unboxing and Review of Soundcore F...,Soundcore,[],"{'Package Dimensions': None, 'Item model numbe...",NaN
1393172,1,Scam,Don't get taken..these are nothing but pure junk.,[],B08CVC7DFH,B08CVC7DFH,AEUWNDP44BOO7IANHVEN65NHQF6A,2020-11-13 23:05:44.958,0,True,...,41,[Titanium frame],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],LINPING,[],{'Package Dimensions': '7 x 3.9 x 2.5 inches; ...,NaN
1401296,3,NOT AS EXPECTED,GOOD SPECS JUST LIKE NO LINES BUT THEY DIDNT G...,[],B08GS3D8Y9,B08GS3D8Y9,AG6NIYQEJ4VPI6OA63HG4VTG4JCQ,2020-11-27 16:29:23.164,0,True,...,17,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],LINPING,[],"{'Package Dimensions': None, 'Item model numbe...",NaN


In [7]:
merged_ai.columns

Index(['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id',
       'timestamp', 'helpful_vote', 'verified_purchase', 'prod_main_category',
       'prod_title', 'prod_average_rating', 'prod_rating_number',
       'prod_features', 'prod_description', 'prod_price', 'prod_images',
       'prod_videos', 'prod_store', 'prod_categories', 'prod_details',
       'prod_bought_together'],
      dtype='object')

### 去掉多余的列

In [ ]:
drop_cols = [
    'prod_description','prod_features','prod_details','prod_bought_together',
    'prod_images','prod_videos','prod_store'
]

df_map = {
    'merged_ai': merged_ai,
    'merged_sunglasses': merged_sunglasses,
    'merged_lens': merged_lens,
    'merged_eyeglasses': merged_eyeglasses
}

cleaned = {}
for name, df in df_map.items():
    cols_to_drop = [c for c in drop_cols if c in df.columns]
    cleaned[f'{name}_clean'] = df.drop(columns=cols_to_drop).copy()


merged_ai_clean = cleaned['merged_ai_clean']
merged_sunglasses_clean = cleaned['merged_sunglasses_clean']
merged_lens_clean = cleaned['merged_lens_clean']
merged_eyeglasses_clean = cleaned['merged_eyeglasses_clean']

In [9]:
merged_ai_clean

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,prod_main_category,prod_title,prod_average_rating,prod_rating_number,prod_price,prod_categories
262501,3,Works,It does work and I do recommend it.,[],B08GQNBN9V,B08GQNBN9V,AFAOAXQ5CC46LHQW5BQZ2O4NPVIQ,2021-03-20 12:06:07.162,0,True,AMAZON FASHION,"Reading Glasses German Smart Glasses, Zoom Pre...",3.3,3,NaN,[]
431466,5,They work wonderfully,Great glasses work brilliantly and for a reaso...,[],B08GS3D8Y9,B08GS3D8Y9,AH2HQBJ4QCZQ5FA7F5Q3H3IHASBA,2020-12-24 18:25:46.654,0,True,AMAZON FASHION,"Reading Glasses German Smart Glasses, Zoom Pre...",3.1,17,NaN,[]
450770,1,Didn’t hold up,Ordered in September and initially loved them....,[],B09Z2V7QV3,B09Z2V7QV3,AFWBQXUKVTEAYACHQHJX6FA7GKHQ,2022-12-16 00:48:36.147,0,False,AMAZON FASHION,TUGAU Smart Audio Glasses with Open-ear Stereo...,2.8,2,NaN,[]
820344,1,must be worn in place,not working well,[],B08GS3D8Y9,B08GS3D8Y9,AEZJEWS7CQ56S7IDHKFFH73XGSHA,2020-11-14 03:05:22.226,0,True,AMAZON FASHION,"Reading Glasses German Smart Glasses, Zoom Pre...",3.1,17,NaN,[]
1021182,5,Looks good,The glasses look good and block sunlight well,[],B09GVF9BMS,B09GVF9BMS,AHGQ7TYVPSM7W4SQPAZJI4YEEYDQ,2022-07-29 20:57:20.832,0,True,AMAZON FASHION,"Soundcore by Anker, Frames Landmark (Tortoise)...",4.1,32,NaN,[]
1200229,2,"Don’t order a combo pack with extra lenses, th...",Sound quality is mediocre at best. Don’t order...,[],B09GVF9BMS,B09GVF9BMS,AE2EW47S5SXBC2FA42NVULKERSPQ,2022-02-16 16:54:13.618,2,True,AMAZON FASHION,"Soundcore by Anker, Frames Landmark (Tortoise)...",4.1,32,NaN,[]
1224551,4,Quize probar por ser algo nuevo.,Precio.Me queda bien.,[],B08GS3D8Y9,B08GS3D8Y9,AGQDPIZ2RQ6FSD6FV7MNUJPIY2KQ,2020-11-28 10:28:15.695,0,True,AMAZON FASHION,"Reading Glasses German Smart Glasses, Zoom Pre...",3.1,17,NaN,[]
1232722,1,Professional product,I am very happy with the product,[],B09GVF9BMS,B09GVF9BMS,AEJUZG7U7AWOYUX3X62MUBXG4EZQ,2022-11-01 03:13:02.864,0,True,AMAZON FASHION,"Soundcore by Anker, Frames Landmark (Tortoise)...",4.1,32,NaN,[]
1393172,1,Scam,Don't get taken..these are nothing but pure junk.,[],B08CVC7DFH,B08CVC7DFH,AEUWNDP44BOO7IANHVEN65NHQF6A,2020-11-13 23:05:44.958,0,True,AMAZON FASHION,"Reading Glasses German Smart Glasses, Zoom Pre...",2.9,41,NaN,[]
1401296,3,NOT AS EXPECTED,GOOD SPECS JUST LIKE NO LINES BUT THEY DIDNT G...,[],B08GS3D8Y9,B08GS3D8Y9,AG6NIYQEJ4VPI6OA63HG4VTG4JCQ,2020-11-27 16:29:23.164,0,True,AMAZON FASHION,"Reading Glasses German Smart Glasses, Zoom Pre...",3.1,17,NaN,[]


### 导出文件

In [12]:
import os

output_dir = "Amazon-Fashion-2023-output-parquet"
os.makedirs(output_dir, exist_ok=True)

# 方法一：逐个保存
merged_ai_clean.to_parquet(os.path.join(output_dir, "merged_ai_clean.parquet"), engine="pyarrow", index=False)
merged_sunglasses_clean.to_parquet(os.path.join(output_dir, "merged_sunglasses_clean.parquet"), engine="pyarrow", index=False)
merged_lens_clean.to_parquet(os.path.join(output_dir, "merged_lens_clean.parquet"), engine="pyarrow", index=False)
merged_eyeglasses_clean.to_parquet(os.path.join(output_dir, "merged_eyeglasses_clean.parquet"), engine="pyarrow", index=False)
